<a href="https://colab.research.google.com/github/NauaneLopes/Citologia_Oral/blob/main/EDA_sele%C3%A7%C3%A3o_FEATURES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("/content/drive/MyDrive/RESULTADOS/EDA/features/f_esp_correta.csv")
df.shape

(65521, 121)

In [4]:
# localizar colunas que possuem nome com _hist_bin_ em qualquer parte
df.filter(regex='_hist_bin_').columns

# Excluir colunas que contenha _hist_bin_
df = df.drop(df.filter(regex='_hist_bin_').columns, axis=1)
df.shape

(65521, 25)

In [5]:
# Filtrar df para normal e abnormal
df_filt = df[df['classe'].isin(['normal', 'abnormal'])]
df_filt.shape

(31025, 25)

In [6]:
df.columns

Index(['conjunto', 'classe', 'nome_imagem', 'R_media', 'R_mediana', 'R_moda',
       'R_desvio', 'R_min', 'R_max', 'G_media', 'G_mediana', 'G_moda',
       'G_desvio', 'G_min', 'G_max', 'B_media', 'B_mediana', 'B_moda',
       'B_desvio', 'B_min', 'B_max', 'R_div_B', 'G_div_B', 'R_div_G',
       'RG_div_2B'],
      dtype='object')

In [7]:
# excluir colunas R_div_B', 'G_div_B', 'R_div_G', 'RG_div_2B'
variaveis_excluir = ['R_div_B', 'G_div_B', 'R_div_G', 'RG_div_2B']
df_filt = df_filt.drop(variaveis_excluir, axis=1)
df_filt.shape

(31025, 21)

In [14]:
import pandas as pd
df_m = pd.read_csv('/content/drive/MyDrive/RESULTADOS/EDA/features/features_morfologicas_normal_abnormal.csv')
df_m.shape

(45911, 14)

In [15]:
df_m.columns

Index(['area', 'perimetro', 'circularidade', 'excentricidade', 'elongacao',
       'compacidade', 'bbox_ratio', 'centroide_x', 'centroide_y', 'orientacao',
       'euler_number', 'nome_imagem', 'classe', 'subset'],
      dtype='object')

In [16]:
# Excluir variáveis
variaveis_excluir = [ 'orientacao','euler_number']
df_m = df_m.drop(variaveis_excluir, axis=1)
df_m.shape

(45911, 12)

In [17]:
! pip install -U kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 4.1 MB/s eta 0:00:00


In [18]:
import plotly.figure_factory as ff
import numpy as np
import plotly.io as pio
import kaleido
# pegar apenas as variáveis númericas
variaveis_corr = df_m.select_dtypes(include=['number']).columns


# Calcular a matriz de correlação
corr = df_m[variaveis_corr].corr().round(2)

# Preparar os dados para o Plotly
z = np.array(corr.values)
x = list(corr.columns)
y = list(corr.index)

# Criar o heatmap interativo
fig = ff.create_annotated_heatmap(
    z,
    x=x,
    y=y,
    annotation_text=corr.values.round(2),
    colorscale= 'YlGnBu',
    showscale=True,
    reversescale=True,
    zmin=-1,
    zmax=1
)

mc = fig.update_layout(
    title='Matriz de Correlação — Variáveis Morfológicas',
    width=1050,
    height=1050,
    xaxis_title="Variáveis",
    yaxis_title="Variáveis"
)

fig.show()



/usr/local/lib/python3.11/dist-packages/kaleido/__init__.py:14: UserWarning: 


This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

  from .kaleido import Kaleido


In [19]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import plotly.express as px

# === 1. Treinar o Random Forest ===
X = df_m.drop(columns=['classe', 'subset', 'nome_imagem'])  # Substitua 'target' pelo nome real da sua variável alvo
y = df_m['classe']


model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# === 2. Calcular a importância das features ===
importancias = model.feature_importances_
df_importancia = pd.DataFrame({
    'Variável': X.columns,
    'Importância': importancias
}).sort_values(by='Importância', ascending=True)

# === 3. Plot com Plotly Express ===
fig = px.bar(
    df_importancia,
    x='Importância',
    y='Variável',
    orientation='h',
    title='Importância das Variáveis - Random Forest',
    color='Importância',
    color_continuous_scale='Blues'  # Pode escolher outras: 'Viridis', 'Cividis', 'Inferno'
)

fig.update_layout(
    xaxis_title='Importância',
    yaxis_title='Variável',
    template='plotly_white',
    title_x=0.5
)

fig.show()

# === 5. Exportar tabela para LaTeX ===
with open('feature_importance_rf.tex', 'w') as f:
    f.write(df_importancia.to_latex(index=False,
                                    caption="Importância das variáveis segundo Random Forest.",
                                    label="tab:feature_importance_rf",
                                    float_format="%.4f"))

In [27]:
# Abrir df_int
import pandas as pd

df_int = pd.read_csv('/content/drive/MyDrive/RESULTADOS/EDA/features/df_int.csv')
df_int.shape

(31025, 21)

In [28]:
df_int.columns

Index(['subset', 'classe', 'nome_imagem', 'relacao_sinal_fundo_R',
       'intensidade_sinal_G', 'relacao_sinal_fundo_G', 'intensidade_sinal_B',
       'relacao_sinal_fundo_B', 'textura_borda_B', 'dif_int_sinal_R_G',
       'dif_int_sinal_R_B', 'dif_int_sinal_G_B', 'indice_sinal_R_G',
       'indice_sinal_R_B', 'indice_sinal_G_B', 'dif_textura_RG',
       'dif_textura_GB', 'dif_textura_RB', 'indice_textura_RG',
       'indice_textura_GB', 'indice_textura_RB'],
      dtype='object')

In [30]:
import plotly.figure_factory as ff
import numpy as np
import plotly.io as pio
import kaleido
# pegar apenas as variáveis númericas
variaveis_corr = df_int.select_dtypes(include=['number']).columns


# Calcular a matriz de correlação
corr = df_int[variaveis_corr].corr().round(2)

# Preparar os dados para o Plotly
z = np.array(corr.values)
x = list(corr.columns)
y = list(corr.index)

# Criar o heatmap interativo
fig = ff.create_annotated_heatmap(
    z,
    x=x,
    y=y,
    annotation_text=corr.values.round(2),
    colorscale= 'YlGnBu',
    showscale=True,
    reversescale=True,
    zmin=-1,
    zmax=1
)

mc = fig.update_layout(
    title='Matriz de Correlação — Variáveis Intensidade',
    width=1500,
    height=1500,

)

fig.show()



In [31]:
import numpy as np

# Filtrar correlações com módulo maior ou igual a 0.85
corr_filtrada = corr[abs(corr) >= 0.85]

# Remover diagonal e duplicados
corr_filtrada = corr_filtrada.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# Empilhar (transforma em Series) e ordenar
corr_series = corr_filtrada.stack().sort_values(ascending=False)

# Converter para DataFrame
tabela_corr = corr_series.reset_index()
tabela_corr.columns = ['Variável 1', 'Variável 2', 'Correlação']

# Exportar para LaTeX
with open("tabela_correlacao_int.tex", "w") as f:
    f.write(tabela_corr.to_latex(index=False,
                                 caption="Pares de Variáveis com Correlação ≥ 0,85.",
                                 label="tab:tab_corr_int",
                                 float_format="%.2f"))


In [32]:
tabela_corr

,Variável 1,Variável 2,Correlação
0,dif_int_sinal_R_G,dif_textura_RG,0.96
1,dif_int_sinal_G_B,dif_textura_GB,0.96
2,indice_textura_GB,indice_textura_RB,0.96
3,dif_int_sinal_R_B,dif_textura_RB,0.95
4,intensidade_sinal_B,textura_borda_B,0.94
5,indice_sinal_R_B,indice_sinal_G_B,0.88


In [33]:
import numpy as np

# Filtrar apenas correlações negativas <= -0.85
corr_filtrada_neg = corr[corr <= -0.60]

# Manter apenas metade superior (evitar diagonais e duplicados)
corr_filtrada_neg = corr_filtrada_neg.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# Remover NaNs e ordenar do mais negativo para o menos negativo
resultado = corr_filtrada_neg.stack().sort_values()

print(resultado)


dif_textura_RG       dif_textura_GB      -0.73
dif_int_sinal_R_G    dif_int_sinal_G_B   -0.68
dif_int_sinal_G_B    dif_textura_RG      -0.68
dif_int_sinal_R_G    dif_textura_GB      -0.67
intensidade_sinal_B  dif_textura_RB      -0.66
                     dif_int_sinal_R_B   -0.64
textura_borda_B      dif_textura_RB      -0.60
dtype: float64


In [34]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import plotly.express as px

# === 1. Treinar o Random Forest ===
X = df_int.drop(columns=['classe', 'subset', 'nome_imagem'])
y = df_int['classe']


model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# === 2. Calcular a importância das features ===
importancias = model.feature_importances_
df_importancia_int = pd.DataFrame({
    'Variável': X.columns,
    'Importância': importancias
}).sort_values(by='Importância', ascending=True)

# === 3. Plot com Plotly Express ===
fig = px.bar(
    df_importancia_int,
    x='Importância',
    y='Variável',
    orientation='h',
    title='Importância das Variáveis - Random Forest',
    color='Importância',
    color_continuous_scale='Blues'  # Pode escolher outras: 'Viridis', 'Cividis', 'Inferno'
)

fig.update_layout(
    xaxis_title='Importância',
    yaxis_title='Variável',
    template='plotly_white',
    title_x=0.5
)

fig.show()

# === 5. Exportar tabela para LaTeX ===
with open('feature_importance_int.tex', 'w') as f:
    f.write(df_importancia_int.to_latex(index=False,
                                    caption="Importância das variáveis segundo Random Forest.",
                                    label="tab:feature_importance_int",
                                    float_format="%.4f"))

In [35]:
# lista de feature importance inferior a 0,0200
# Supondo que df_importancia seja o DataFrame com as importâncias
# Exemplo de estrutura: ['Variável', 'Importância']

# === 1. Definir o limiar de corte ===
limiar_importancia = 0.0200

# === 2. Criar lista de variáveis a excluir ===
variaveis_para_excluir = df_importancia_int[df_importancia_int['Importância'] <= limiar_importancia]['Variável'].tolist()

print("Variáveis a excluir:", variaveis_para_excluir)

# Excluir essas variáveis do dataset df_int
df_int = df_int.drop(columns=variaveis_para_excluir)
df_int.shape

Variáveis a excluir: []


(31025, 21)

In [36]:
df_int.columns

Index(['subset', 'classe', 'nome_imagem', 'relacao_sinal_fundo_R',
       'intensidade_sinal_G', 'relacao_sinal_fundo_G', 'intensidade_sinal_B',
       'relacao_sinal_fundo_B', 'textura_borda_B', 'dif_int_sinal_R_G',
       'dif_int_sinal_R_B', 'dif_int_sinal_G_B', 'indice_sinal_R_G',
       'indice_sinal_R_B', 'indice_sinal_G_B', 'dif_textura_RG',
       'dif_textura_GB', 'dif_textura_RB', 'indice_textura_RG',
       'indice_textura_GB', 'indice_textura_RB'],
      dtype='object')

In [37]:
# colunas df_int para excluir
var_excluir = ['dif_int_sinal_R_B']
df_int = df_int.drop(columns=var_excluir)
df_int.shape

(31025, 20)

In [39]:
# salvar df_int como csv no drive
df_int.to_csv('/content/drive/MyDrive/RESULTADOS/EDA/features/df_int.csv', index=False)
df_int.head()

,subset,classe,nome_imagem,relacao_sinal_fundo_R,intensidade_sinal_G,relacao_sinal_fundo_G,intensidade_sinal_B,relacao_sinal_fundo_B,textura_borda_B,dif_int_sinal_R_G,dif_int_sinal_G_B,indice_sinal_R_G,indice_sinal_R_B,indice_sinal_G_B,dif_textura_RG,dif_textura_GB,dif_textura_RB,indice_textura_RG,indice_textura_GB,indice_textura_RB
0,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x100907-1...,0.661115,25.926083,0.404919,16.761257,0.314461,35.134921,25.854715,9.164826,1.997247,3.089315,1.546786,19.246032,9.972222,29.218254,1.426673,1.283826,1.831601
1,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x100912-1...,0.549172,33.670210,0.352743,27.139944,0.316112,56.811352,26.716858,6.530267,1.793487,2.225026,1.240615,19.858097,8.237062,28.095159,1.305282,1.144990,1.494534
2,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x100951-1...,0.706284,66.505037,0.543080,55.120879,0.501585,83.089674,31.602564,11.384158,1.475191,1.779863,1.206531,22.831522,11.355978,34.187500,1.241742,1.136671,1.411453
3,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x100978-1...,0.576016,4.024803,0.150319,1.175197,0.074456,6.630385,22.441732,2.849606,6.575858,22.520938,3.424791,21.682540,9.043084,30.725624,2.383390,2.363882,5.634054
4,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x102335-1...,0.561901,52.007347,0.433462,41.100477,0.382979,73.929379,16.724384,10.906871,1.321577,1.672286,1.265371,9.338983,11.995763,21.334746,1.108687,1.162260,1.288583


In [40]:
# Correlação para estatísticas de cor
import plotly.figure_factory as ff
import numpy as np
import plotly.io as pio
import kaleido
# pegar apenas as variáveis númericas
variaveis_corr = df_filt.select_dtypes(include=['number']).columns


# Calcular a matriz de correlação
corr = df_filt[variaveis_corr].corr().round(2)

# Preparar os dados para o Plotly
z = np.array(corr.values)
x = list(corr.columns)
y = list(corr.index)

# Criar o heatmap interativo
fig = ff.create_annotated_heatmap(
    z,
    x=x,
    y=y,
    annotation_text=corr.values.round(2),
    colorscale= 'YlGnBu',
    showscale=True,
    reversescale=True,
    zmin=-1,
    zmax=1
)

mc = fig.update_layout(
    title='Matriz de Correlação — Variáveis Intensidade',
    width=1000,
    height=1000,

)

fig.show()



In [41]:
import numpy as np

# Filtrar correlações com módulo maior ou igual a 0.85
corr_filtrada = corr[abs(corr) >= 0.85]

# Remover diagonal e duplicados
corr_filtrada = corr_filtrada.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# Empilhar (transforma em Series) e ordenar
corr_series = corr_filtrada.stack().sort_values(ascending=False)

# Converter para DataFrame
tabela_corr = corr_series.reset_index()
tabela_corr.columns = ['Variável 1', 'Variável 2', 'Correlação']

# Exportar para LaTeX
with open("tabela_correlacao_est.tex", "w") as f:
    f.write(tabela_corr.to_latex(index=False,
                                 caption="Pares de Variáveis com Correlação ≥ 0,85.",
                                 label="tab:tab_corr_est",
                                 float_format="%.2f"))


In [42]:
tabela_corr

,Variável 1,Variável 2,Correlação
0,R_media,R_mediana,1.00
1,B_media,B_mediana,1.00
2,G_media,G_mediana,1.00
3,B_mediana,B_moda,0.95
4,R_mediana,R_moda,0.95
5,G_mediana,G_moda,0.94
6,R_media,R_moda,0.94
7,B_media,B_moda,0.94
8,G_media,G_moda,0.93
9,R_media,R_max,0.92


In [43]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import plotly.express as px

# === 1. Treinar o Random Forest ===
X = df_filt.drop(columns=['classe', 'conjunto', 'nome_imagem'])
y = df_filt['classe']


model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# === 2. Calcular a importância das features ===
importancias = model.feature_importances_
df_importancia_int = pd.DataFrame({
    'Variável': X.columns,
    'Importância': importancias
}).sort_values(by='Importância', ascending=True)

# === 3. Plot com Plotly Express ===
fig = px.bar(
    df_importancia_int,
    x='Importância',
    y='Variável',
    orientation='h',
    title='Importância das Variáveis - Random Forest',
    color='Importância',
    color_continuous_scale='Blues'  # Pode escolher outras: 'Viridis', 'Cividis', 'Inferno'
)

fig.update_layout(
    xaxis_title='Importância',
    yaxis_title='Variável',
    template='plotly_white',
    title_x=0.5
)

fig.show()

# === 5. Exportar tabela para LaTeX ===
with open('feature_importance_est_b.tex', 'w') as f:
    f.write(df_importancia_int.to_latex(index=False,
                                    caption="Importância das variáveis segundo Random Forest.",
                                    label="tab:feature_importance_est_b",
                                    float_format="%.4f"))

In [44]:
# variáveis a serem excluídas df_filter
var_filt = ['R_mediana', 'B_mediana', 'G_mediana', 'R_moda', 'G_moda', 'B_moda', 'R_max', 'G_max', 'B_desvio', 'B_max', 'G_desvio']

df_filt = df_filt.drop(columns=var_filt)
df_filt.shape

(31025, 10)

In [46]:
# mudar nome coluna conjunto
df_filt = df_filt.rename(columns={'conjunto': 'subset'})

df_filt.columns
# salvar df_filt como csv no drive
df_filt.to_csv('/content/drive/MyDrive/RESULTADOS/EDA/features/df_filt.csv', index=False)

In [47]:
df_m.head()

,area,perimetro,circularidade,excentricidade,elongacao,compacidade,bbox_ratio,centroide_x,centroide_y,nome_imagem,classe,subset
0,2376.0,227.480231,0.576991,0.570497,1.217582,0.595489,0.814286,39.852273,24.619529,bbox_3_2020_11_30__12_37__0825_b0s0c0x499200-1...,normal,train
1,34573.0,1665.230627,0.156674,0.787662,1.623094,0.640051,0.824219,123.679924,164.630203,center_0_2020_11_30__12_37__0825_b0s0c0x315200...,normal,train
2,241.0,70.941125,0.601770,0.816937,1.733924,0.557870,1.687500,247.053942,13.240664,center_0_2020_11_30__12_37__0825_b0s0c0x315200...,normal,train
3,29393.0,750.261977,0.656187,0.306997,1.050740,0.887389,0.989071,85.599122,167.709489,center_1_2020_11_30__12_37__0825_b0s0c0x278400...,normal,train
4,228.0,94.349242,0.321861,0.986005,5.998323,0.581633,0.163265,208.622807,252.587719,center_1_2020_11_30__12_37__0825_b0s0c0x278400...,normal,train


In [49]:
# Agrupar pelas imagens e calcular a média de todas as variáveis numéricas
df_morf_at = df_m.groupby(['nome_imagem', 'classe', 'subset']).mean(numeric_only=True).reset_index()
# reordenar colunas ('nome_imagem', 'classe', 'subset') para subset, classe e nome imagem
df_morf_at = df_morf_at[['subset', 'classe', 'nome_imagem'] + [col for col in df_morf_at.columns if col not in ['subset', 'classe', 'nome_imagem']]]
print(df_morf_at.shape)
df_morf_at.head()
# salvar df_morf_at como csv no drive
df_morf_at.to_csv('/content/drive/MyDrive/RESULTADOS/EDA/features/df_morf_at.csv', index=False)

(31025, 12)


In [50]:
# características a serem excluídas
var_excluir = ['perimetro', 'excentricidade']
df_morf_at = df_morf_at.drop(columns=var_excluir)
df_morf_at.shape

(31025, 10)

In [51]:
print(df_filt.columns)
print(df_morf_at.columns)
print(df_int.columns)

Index(['subset', 'classe', 'nome_imagem', 'R_media', 'R_desvio', 'R_min',
       'G_media', 'G_min', 'B_media', 'B_min'],
      dtype='object')
Index(['subset', 'classe', 'nome_imagem', 'area', 'circularidade', 'elongacao',
       'compacidade', 'bbox_ratio', 'centroide_x', 'centroide_y'],
      dtype='object')
Index(['subset', 'classe', 'nome_imagem', 'relacao_sinal_fundo_R',
       'intensidade_sinal_G', 'relacao_sinal_fundo_G', 'intensidade_sinal_B',
       'relacao_sinal_fundo_B', 'textura_borda_B', 'dif_int_sinal_R_G',
       'dif_int_sinal_G_B', 'indice_sinal_R_G', 'indice_sinal_R_B',
       'indice_sinal_G_B', 'dif_textura_RG', 'dif_textura_GB',
       'dif_textura_RB', 'indice_textura_RG', 'indice_textura_GB',
       'indice_textura_RB'],
      dtype='object')


In [ ]:
# Script união df_morf e df_espaciais
# === 1. Padronizar identificadores ===
for df in [df_int, df_morf_at, df_filt]:
    df['subset'] = df['subset'].astype(str).str.strip().str.lower()
    df['classe'] = df['classe'].astype(str).str.strip().str.lower()
    df['nome_imagem'] = df['nome_imagem'].astype(str).str.strip().str.lower()

# === 2. Ordenar os três arquivos pela chave (garante alinhamento) ===
chaves = ['subset', 'classe', 'nome_imagem']
df_int = df_int.sort_values(by=chaves).reset_index(drop=True)
df_morf_at = df_morf_at.sort_values(by=chaves).reset_index(drop=True)
df_filt = df_filt.sort_values(by=chaves).reset_index(drop=True)



# === 3. Concatenar apenas as colunas de features ===
df_final = pd.concat([
    df_int,
    df_morf_at.drop(columns=chaves),
    df_filt.drop(columns=chaves)
], axis=1)

# === 4. Salvar resultado ===
df_final.to_csv("/content/drive/MyDrive/RESULTADOS/features/ARQUIVOS_UNIDOS_REDUCAO/df_GERAL_PESOS.csv", index=False)
print(f"Arquivo final salvo com {df_final.shape[0]} linhas e {df_final.shape[1]} colunas.")

Arquivo final salvo com 31025 linhas e 39 colunas.


In [ ]:
df_new = pd.read_csv('/content/drive/MyDrive/RESULTADOS/features/ARQUIVOS_UNIDOS_REDUCAO/df_GERAL_PESOS.csv')
df_new.shape

(31025, 39)

In [ ]:
# Excluir colunas
excluir_colunas = ['R_div_B', 'G_div_B', 'R_div_G',
       'RG_div_2B', 'dif_int_sinal_R_G', 'indice_textura_GB', 'dif_textura_GB', 'dif_int_sinal_R_B', 'textura_borda_B', 'indice_sinal_G_B']
df_new = df_new.drop(columns=excluir_colunas)
df_new.shape

(31025, 29)

In [ ]:
# Correlação para estatísticas de cor
import plotly.figure_factory as ff
import numpy as np
import plotly.io as pio
import kaleido
# pegar apenas as variáveis númericas
variaveis_corr = df_new.select_dtypes(include=['number']).columns


# Calcular a matriz de correlação
corr = df_new[variaveis_corr].corr().round(2)

# Preparar os dados para o Plotly
z = np.array(corr.values)
x = list(corr.columns)
y = list(corr.index)

# Criar o heatmap interativo
fig = ff.create_annotated_heatmap(
    z,
    x=x,
    y=y,
    annotation_text=corr.values.round(2),
    colorscale= 'YlGnBu',
    showscale=True,
    reversescale=True,
    zmin=-1,
    zmax=1
)

mc = fig.update_layout(
    title='Matriz de Correlação — Variáveis Intensidade',
    width=1500,
    height=1500,

)

fig.show()



In [ ]:
import numpy as np

# Filtrar correlações com módulo maior ou igual a 0.85
corr_filtrada = corr[abs(corr) >= 0.85]

# Remover diagonal e duplicados
corr_filtrada = corr_filtrada.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# Empilhar (transforma em Series) e ordenar
corr_series = corr_filtrada.stack().sort_values(ascending=False)

# Converter para DataFrame
tabela_corr = corr_series.reset_index()
tabela_corr.columns = ['Variável 1', 'Variável 2', 'Correlação']

# Exportar para LaTeX
with open("tabela_correlacao_est.tex", "w") as f:
    f.write(tabela_corr.to_latex(index=False,
                                 caption="Pares de Variáveis com Correlação ≥ 0,85.",
                                 label="tab:tab_corr_est",
                                 float_format="%.2f"))


In [ ]:
tabela_corr

,Variável 1,Variável 2,Correlação
0,intensidade_sinal_G,G_media,1.0
1,intensidade_sinal_B,R_media,1.0


In [ ]:
df_importancia_int


,Variável,Importância
5,intensidade_fundo_G,0.014787
3,textura_borda_R,0.015335
0,intensidade_sinal_R,0.016242
7,textura_borda_G,0.016246
9,intensidade_fundo_B,0.017925
1,intensidade_fundo_R,0.018620
11,textura_borda_B,0.020052
17,indice_sinal_G_B,0.020542
22,indice_textura_GB,0.021144
4,intensidade_sinal_G,0.025561


In [ ]:
# Salvar df_new em csv no drive
df_new.to_csv('/content/drive/MyDrive/RESULTADOS/features/df_new.csv', index=False)
df_new.head()

,subset,classe,nome_imagem,relacao_sinal_fundo_R,intensidade_sinal_G,relacao_sinal_fundo_G,intensidade_sinal_B,relacao_sinal_fundo_B,dif_int_sinal_G_B,indice_sinal_R_G,...,bbox_ratio,centroide_x,centroide_y,R_media,R_desvio,R_min,G_media,G_min,B_media,B_min
0,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x100907-1...,0.661115,25.926083,0.404919,16.761257,0.314461,9.164826,1.997247,...,1.054054,22.457944,20.989805,16.761257,9.487546,0,25.926083,8,51.780799,38
1,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x100912-1...,0.549172,33.670210,0.352743,27.139944,0.316112,6.530267,1.793487,...,0.784091,49.802994,36.213495,27.139944,16.431167,0,33.670210,3,60.387069,33
2,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x100951-1...,0.706284,66.505037,0.543080,55.120879,0.501585,11.384158,1.475191,...,0.982143,30.243132,28.446429,55.120879,15.120912,28,66.505037,39,98.107601,79
3,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x100978-1...,0.576016,4.024803,0.150319,1.175197,0.074456,2.849606,6.575858,...,1.070175,30.816568,37.432347,1.175197,1.824488,0,4.024803,0,26.466535,16
4,test,abnormal,bbox_0_2019_07_10__14_48__0045_b0s0c0x102335-1...,0.561901,52.007347,0.433462,41.100477,0.382979,10.906871,1.321577,...,0.945652,45.657863,44.777998,41.100477,13.344247,17,52.007347,25,68.731732,46
